# 第7章 资格迹

## 目录
1. [环境搭建](#1-环境搭建)
2. [资格迹定义](#2-资格迹定义)
3. [TD(λ)算法](#3-TDλ算法)
4. [SARSA(λ)算法](#4-SARSAλ算法)
5. [编程题](#5-编程题)

---

## 1. 环境搭建

In [ ]:
!pip install numpy matplotlib -q
import numpy as np
import matplotlib.pyplot as plt
import random
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("环境搭建完成!")

---

## 2. 资格迹定义

**资格迹（Eligibility Traces）**是为每个状态-动作对维护的"迹值"，用于记录近期交互中的活跃度。


**更新公式**：
$$E_t(s,a) = \begin{cases} 1 & \text{若 } S_t=s, A_t=a \\ \gamma \lambda E_{t-1}(s,a) & \text{其他} \end{cases}$$

**λ-回报**：
$$G_t^\lambda = (1-\lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$$

- λ=0: 退化为TD(0)
- λ=1: 退化为蒙特卡洛

In [ ]:
class RandomWalkEnv:
    """随机游走环境"""
    def __init__(self, n_states=7):
        self.n_states = n_states
        self.start = n_states // 2
        self.current = self.start
    
    def reset(self):
        self.current = self.start
        return self.current
    
    def step(self, action):
        if action == 0: self.current = max(0, self.current - 1)
        else: self.current = min(self.n_states - 1, self.current + 1)
        done = self.current in [0, self.n_states - 1]
        reward = 1 if self.current == self.n_states - 1 else 0
        return self.current, reward, done

class EligibilityTrace:
    """资格迹"""
    def __init__(self, n_states, gamma=0.9, lamda=0.9, trace_type='accumulating'):
        self.n_states = n_states
        self.gamma = gamma
        self.lamda = lamda
        self.trace_type = trace_type
        self.traces = np.zeros(n_states)
    
    def reset(self):
        self.traces = np.zeros(self.n_states)
    
    def update(self, state):
        if self.trace_type == 'accumulating':
            self.traces *= self.gamma * self.lamda
            self.traces[state] += 1
        else:  # replacing
            self.traces *= self.gamma * self.lamda
            self.traces[state] = 1
    
    def get_traces(self):
        return self.traces.copy()

# 测试资格迹
print("测试资格迹...")
trace = EligibilityTrace(7, gamma=0.9, lamda=0.9)
for _ in range(5):
    trace.reset()
    trace.update(3)
    trace.update(2)
    trace.update(1)
    print(f"  迹值: {trace.get_traces()}")

---

## 3. TD(λ)算法

In [ ]:
class TDLambda:
    """TD(λ)算法"""
    def __init__(self, n_states, alpha=0.1, gamma=0.9, lamda=0.9):
        self.n_states = n_states
        self.alpha = alpha
        self.gamma = gamma
        self.lamda = lamda
        self.values = np.zeros(n_states)
        self.traces = EligibilityTrace(n_states, gamma, lamda)
    
    def reset(self):
        self.traces.reset()
    
    def update(self, state, reward, next_state, done):
        if done: target = reward
        else: target = reward + self.gamma * self.values[next_state]
        td_error = target - self.values[state]
        self.traces.update(state)
        self.values += self.alpha * td_error * self.traces.get_traces()
    
    def get_values(self): return self.values.copy()

def compare_lambda():
    """对比不同λ值"""
    lambdas = [0.0, 0.3, 0.6, 0.9, 1.0]
    n_episodes = 500
    true_values = np.linspace(0.5, 1.0, 7)
    
    plt.figure(figsize=(12, 5))
    for lamda in lambdas:
        errors = []
        td = TDLambda(7, alpha=0.1, gamma=0.9, lamda=lamda)
        for _ in range(n_episodes):
            td.reset()
            env = RandomWalkEnv(7)
            state = env.reset()
            done = False
            while not done:
                action = random.randint(0, 1)
                next_state, reward, done = env.step(action)
                td.update(state, reward, next_state, done)
                state = next_state
            if _ % 50 == 0:
                errors.append(np.mean(np.abs(td.get_values() - true_values)))
        plt.plot(errors, label=f'λ={lamda}', linewidth=2)
    plt.xlabel('回合数(×50)')
    plt.ylabel('误差')
    plt.title('TD(λ)不同λ值对比')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('td_lambda.png', dpi=150)
    plt.show()

compare_lambda()

---

## 4. SARSA(λ)算法

In [ ]:
class CliffWalkingEnv:
    """悬崖漫步环境"""
    def __init__(self):
        self.rows, self.cols = 4, 12
        self.start, self.goal = (3, 0), (3, 11)
        self.cliff = [(3, i) for i in range(1, 11)]
        self.state = self.start
    def reset(self): self.state = self.start; return self.state
    def step(self, action):
        dr, dc = [(-1,0), (1,0), (0,-1), (0,1)][action]
        new_r = max(0, min(self.rows-1, self.state[0]+dr))
        new_c = max(0, min(self.cols-1, self.state[1]+dc))
        new_state = (new_r, new_c)
        if new_state in self.cliff: reward, done = -100, True
        elif new_state == self.goal: reward, done = 0, True
        else: reward, done = -1, False
        self.state = new_state
        return new_state, reward, done
    def get_idx(self, s): return s[0]*self.cols + s[1]

class SARSALambda:
    """SARSA(λ)"""
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=1.0, lamda=0.9, epsilon=0.1):
        self.n_states, self.n_actions = n_states, n_actions
        self.alpha, self.gamma, self.lamda = alpha, gamma, lamda
        self.epsilon = epsilon
        self.q_table = np.zeros((n_states, n_actions))
        self.traces = np.zeros((n_states, n_actions))
    def choose_action(self, s):
        return random.randint(0, self.n_actions-1) if random.random()<self.epsilon else np.argmax(self.q_table[s])
    def reset(self): self.traces = np.zeros((self.n_states, self.n_actions))
    def update(self, s, a, r, ns, na):
        target = r + self.gamma * self.q_table[ns, na]
        td_err = target - self.q_table[s, a]
        self.traces[s,a] += 1
        self.q_table += self.alpha * td_err * self.traces
        self.traces *= self.gamma * self.lamda
    def get_values(self): return self.q_table.copy()

def sarsa_lambda_demo():
    """SARSA(λ)演示"""
    for lamda in [0.0, 0.5, 0.9]:
        rewards = []
        agent = SARSALambda(48, 4, alpha=0.1, gamma=1.0, lamda=lamda)
        for _ in range(300):
            agent.reset()
            env = CliffWalkingEnv()
            s = env.reset()
            s_idx = env.get_idx(s)
            a = agent.choose_action(s_idx)
            total_r = 0
            done = False
            while not done:
                ns, r, done = env.step(a)
                ns_idx = env.get_idx(ns)
                na = agent.choose_action(ns_idx)
                agent.update(s_idx, a, r, ns_idx, na)
                total_r += r
                s_idx, a = ns_idx, na
            rewards.append(total_r)
        print(f"λ={lamda}: 平均奖励={np.mean(rewards[-50:]):.1f}")

sarsa_lambda_demo()

---

## 5. 编程题

### 编程题1
实现TD(λ)算法，验证不同λ取值的效果

In [ ]:
"""编程题1：TD(λ)验证"""
def pe1():
    for lamda in [0.0, 0.5, 1.0]:
        td = TDLambda(7, alpha=0.1, gamma=0.9, lamda=lamda)
        for _ in range(500):
            td.reset()
            env = RandomWalkEnv(7)
            s = env.reset()
            done = False
            while not done:
                a = random.randint(0, 1)
                ns, r, done = env.step(a)
                td.update(s, r, ns, done)
                s = ns
        print(f"λ={lamda}: {td.get_values()}")
pe1()

### 编程题2
比较累积型与替换型资格迹

In [ ]:
"""编程题2：资格迹类型对比"""
def pe2():
    for trace_type in ['accumulating', 'replacing']:
        td = TDLambda(7, alpha=0.1, gamma=0.9, lamda=0.9)
        td.traces.trace_type = trace_type
        for _ in range(500):
            td.reset()
            env = RandomWalkEnv(7)
            s = env.reset()
            done = False
            while not done:
                a = random.randint(0, 1)
                ns, r, done = env.step(a)
                td.update(s, r, ns, done)
                s = ns
        print(f"{trace_type}: {td.get_values()}")
pe2()

### 编程题3
SARSA(λ)在悬崖漫步中分析

In [ ]:
"""编程题3：SARSA(λ)分析"""
def pe3():
    for lamda in [0.0, 0.3, 0.9]:
        agent = SARSALambda(48, 4, alpha=0.1, gamma=1.0, lamda=lamda)
        steps = []
        for _ in range(300):
            agent.reset()
            env = CliffWalkingEnv()
            s = env.reset()
            a = agent.choose_action(env.get_idx(s))
            step = 0
            done = False
            while not done:
                ns, r, done = env.step(a)
                na = agent.choose_action(env.get_idx(ns))
                agent.update(env.get_idx(s), a, r, env.get_idx(ns), na)
                s, a = ns, na
                step += 1
            steps.append(step)
        print(f"λ={lamda}: 平均步数={np.mean(steps[-50:]):.0f}")
pe3()

print("\n第7章 资格迹 - 内容完成!")